# Sample 05: スキーマ先行 Builder & 自動リーダー (`Builder`, `builder.read`)

実データ不要の「プロトコル事前設計」、ドキュメント章立て (`add_document`)、Mermaid 仕様書生成、多言語コード一括出力 (C, Rust, Modern C++, C#, Go)、およびスキーマ駆動の「自動デシリアライズ」を学びます。

### 学べる内容
- `Builder` による仕様ファーストのプロトコル設計
- `add_document` による背景・章立てテキストの追加
- `section` / `caption` による論理階層化と Mermaid `subgraph` 自動生成
- `add_choice` による多態バリアント分岐定義
- 多言語コードエクスポート (`write_c_header`, `write_rust`, `write_cpp`, `write_csharp`, `write_go`)
- `builder.read(data)` によるスキーマ駆動の自動パース
- スキーマ相関デバッグダンプ (`builder.hexdump`, `builder.dump`)

In [1]:
from pathlib import Path

from binary_master import (
    BinaryWriter,
    Builder,
    FixedArray,
    Float32,
    UInt8,
    UInt16,
    UInt32,
    binary_struct,
)

## 1. プロトコル構成構造体の定義

In [2]:
@binary_struct(endian="little")
class PacketHeader:
    magic: UInt32         # シグネチャ: 0x4D534750 ('MSGP')
    version: UInt16       # バージョン
    msg_type: UInt16      # 1 = Text, 2 = Sensor
    payload_size: UInt32  # ペイロード長
    flags: UInt16         # Bit 0: Checksum Footer 有無

@binary_struct
class TextMessage:
    encoding: UInt16
    text_len: UInt16
    content: FixedArray[UInt8, 16]

@binary_struct
class SensorReport:
    sensor_id: UInt32
    temperature: Float32
    pressure: Float32
    humidity: Float32

@binary_struct
class ChecksumFooter:
    crc32: UInt32

## 2. `Builder` スキーマの構築

ダミーデータを作ることなく、プロトコルの構造・章立て・分岐条件を純粋に宣言します。

In [3]:
builder = Builder(
    title="Network Telemetry Protocol Specification",
    version="1.0.0",
    default_endian="little",
    description="テキスト通信と環境テレメトリ計測パケットを統合したバイナリプロトコル仕様。",
)

# 概要ドキュメントの追加
builder.add_document(
    "プロトコル概要とスコープ",
    """本仕様書はネットワークテレメトリプロトコル (NTP-v1) を規定します。
すべての数値フィールドはリトルエンディアンでエンコードされます。

パケットは固定14バイトの `PacketHeader` から開始し、`msg_type` に応じてペイロードが分岐します：
- `0x0001`: `TextMessage`
- `0x0002`: `SensorReport`

また、`flags` の Bit 0 が立っている場合、末尾に 4 バイトの `ChecksumFooter` が付与されます。""",
)

with builder.section("Header Section", "コンテナ識別固定ヘッダー"):
    builder.add_struct(PacketHeader, name="header", desc="固定14バイトパケットヘッダー")

with builder.caption("Payload Section", "動的ペイロードブロック"):
    builder.add_choice(
        name="payload",
        tag_field="msg_type",
        variants={
            1: (TextMessage, "プレーンテキストメッセージ"),
            2: (SensorReport, "マルチチャネル環境センサー計測値"),
        },
        desc="PacketHeader.msg_type による動的分岐",
    )

with builder.caption("Footer Section", "オプション完全性検証ブロック"):
    builder.add_struct(
        ChecksumFooter,
        name="footer",
        desc="CRC32 チェックサム",
        condition="flags & 0x01 != 0",
    )

print("Builder スキーマ構築完了！")

Builder スキーマ構築完了！

## 3. 仕様書 Markdown (Mermaid図付き) および多言語コードの生成

In [4]:
sample_dir = Path("/home/ishii/PycharmProjects/binary_master/sample")

# 1. 仕様書 Markdown (Mermaid パケット図 & フローチャート付き)
spec_path = sample_dir / "telemetry_protocol_spec.md"
builder.write(spec_path, diagram_direction="TD")
print(f"仕様書保存: {spec_path.name}")

# 2. 多言語定義ファイルのエクスポート
builder.write_c_header(sample_dir / "telemetry_protocol.h")
builder.write_rust(sample_dir / "telemetry_protocol.rs")
builder.write_cpp(sample_dir / "telemetry_protocol.hpp")
builder.write_csharp(sample_dir / "telemetry_protocol.cs", namespace="TelemetryProtocol")
builder.write_go(sample_dir / "telemetry_protocol.go", package_name="telemetry")

print("C, Rust, C++, C#, Go のヘッダー・ソース生成完了！")

仕様書保存: telemetry_protocol_spec.md
C, Rust, C++, C#, Go のヘッダー・ソース生成完了！

## 4. スキーマ駆動の自動デシリアライズ (`builder.read`)

受信したバイト列を `builder.read(data)` に渡すだけで、ヘッダーのタグ値や条件分岐をスキーマに従って自動評価し、構造体オブジェクトとして一括復元します。手動で `if/elif` パーサーを書く必要がありません。

In [5]:
# テスト用バイナリの作成（Packet A: SensorReport, Checksum 付き）
w = BinaryWriter()
w.write_struct(PacketHeader(magic=0x5047534D, version=1, msg_type=2, payload_size=16, flags=1))
w.write_struct(SensorReport(sensor_id=101, temperature=23.5, pressure=1013.25, humidity=48.0))
w.write_struct(ChecksumFooter(crc32=0xDEADBEEF))
sensor_binary = w.to_bytes()

# builder.read(data) で自動パース！
res = builder.read(sensor_binary)

print(f"Header magic:    0x{res.header.magic:08X}")
print(f"Payload Type:    {type(res.payload).__name__}")
print(f"Temperature:     {res.payload.temperature:.1f} °C")
print(f"Pressure:        {res.payload.pressure:.2f} hPa")
print(f"Footer CRC32:    0x{res.footer.crc32:08X}")

assert isinstance(res.payload, SensorReport)
assert res.footer.crc32 == 0xDEADBEEF
print("\n自動デシリアライズ成功！")

Header magic:    0x5047534D
Payload Type:    SensorReport
Temperature:     23.5 °C
Pressure:        1013.25 hPa
Footer CRC32:    0xDEADBEEF

自動デシリアライズ成功！

## 5. スキーマ相関デバッグダンプ (`builder.hexdump`, `builder.dump`)

In [6]:
print("--- Schema Correlated Hexdump ---")
print(builder.hexdump(sensor_binary)[:400] + "\n  ...")

print("\n--- Decoded Layout Table ---")
print(builder.dump(sensor_binary, format="table"))
print("Builder と自動リーダーの全検証成功！")

--- Schema Correlated Hexdump ---
Offset    00 01 02 03 04 05 06 07  08 09 0A 0B 0C 0D 0E 0F  |     ASCII      |  Field Annotations
-------------------------------------------------------------------------------------------------
00000000  4d 53 47 50 01 00 02 00  10 00 00 00 01 00 65 00  |MSGP..........e.|  magic=0x5047534D (UInt32); version=1 (UInt16); msg_type=2 (UInt16); payload_size=0x10 (UInt32); flags=1 (UInt16); sensor_id=
  ...

--- Decoded Layout Table ---
+--------+------+--------------+---------+--------+-------------+-----------------+-----------------+
| Offset | Size | Field Name   | Type    | Endian | Hex Bytes   | Value / Preview | Caption         |
+========+======+==============+=========+========+=============+=================+=================+
| 0x0000 |   4B | magic        | UInt32  | Little | 4d 53 47 50 | 0x5047534D      | Header Section  |
| 0x0004 |   2B | version      | UInt16  | Little | 01 00       | 1               | Header Section  |
| 0x0006 |   2B | m